In [1]:
# import optuna
# from optuna.samplers import TPESampler
import warnings

import random
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import re
from collections import defaultdict
import os
import math
from collections import defaultdict
import seaborn as sns
from catboost import CatBoostClassifier, CatBoostRegressor
from lightgbm import LGBMClassifier
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import mutual_info_regression
from sklearn.linear_model import Lasso, Ridge, RidgeCV
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    make_scorer,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from xgboost import XGBClassifier
from pathlib import Path

warnings.filterwarnings("ignore")

In [11]:
# radiology_phenotype_extractor.py

# ---------------- CONFIG ----------------
INPUT_CSV = r"E:/Chrome Dls/MIMIC_IV_Note/note/radiology.csv"   # adjust to actual path
OUTPUT_CSV = "../data/processed/radiology_phenotypes_by_subject.csv"
CHUNKSIZE = 50000
TEXT_COL = "text"        # change if radiology file uses 'report' or 'report_text'
SUBJ_COL = "subject_id"
HADM_COL = "hadm_id"     # optional, used if present
CHARTTIME_COL = "charttime"
STORETIME_COL = "storetime"

# define phenotype patterns (example list — extend as needed)
PHENOTYPE_PATTERNS = {
    "cardiomegaly": [r"\bcardiomegaly\b", r"\benlarged heart\b"],
    "pulmonary_edema": [r"\bpulmonary edema\b", r"\bedema in the lungs\b", r"\binterstitial edema\b"],
    "congestive_heart_failure": [r"\bcongestive heart failure\b", r"\bcongestive hf\b", r"\bchf\b"],
    "left_ventricular_hypertrophy": [r"\bleft ventricular hypertrophy\b", r"\blvh\b", r"\bventricular hypertrophy\b"],
    "aortic_atherosclerosis": [r"\baortic atheroscleros", r"\baortic atherosclerosis\b", r"\baortic athero\b"],
    "pulmonary_hypertension_suggestion": [r"\bpulmonary hypertension\b", r"\bpulmonary hypertens\b"]
    # add more phenotypes/patterns or synonyms as needed
}

# negation tokens (simple heuristic); window in chars before match
NEGATION_PATTERN = re.compile(r"\b(no|denies|denied|without|not|negative for|ruled out|rule out|absence of|free of)\b", flags=re.IGNORECASE)
NEGATION_WINDOW = 120

# ------------------------------------------------

# compile phenotype regexes
phen_regexes = {ph: [re.compile(p, flags=re.IGNORECASE) for p in pats] for ph, pats in PHENOTYPE_PATTERNS.items()}

def is_negated_around(text, match_start, window=NEGATION_WINDOW):
    if not isinstance(text, str) or text == "":
        return False
    start = max(0, match_start - window)
    return bool(NEGATION_PATTERN.search(text[start:match_start]))

# container: nested dict: stats[subject][phenotype] -> dict of counters/times
stats = defaultdict(lambda: defaultdict(lambda: {
    "any_mention": False,
    "mention_count": 0,
    "first_mention_time": pd.NaT,
    "last_mention_time": pd.NaT
}))

# helper to update subject-phenotype stats
def update_pheno_stats(subj, pheno, mention_time):
    entry = stats[subj][pheno]
    entry["any_mention"] = True
    entry["mention_count"] += 1
    if pd.isna(entry["first_mention_time"]) or (not pd.isna(mention_time) and mention_time < entry["first_mention_time"]):
        entry["first_mention_time"] = mention_time
    if pd.isna(entry["last_mention_time"]) or (not pd.isna(mention_time) and mention_time > entry["last_mention_time"]):
        entry["last_mention_time"] = mention_time

# read in chunks; try to limit memory by only selecting columns we need
usecols = [SUBJ_COL, CHARTTIME_COL, STORETIME_COL, TEXT_COL]
# if hadm col present, include
try:
    # attempt to detect if hadm column exists (safe fallback)
    sample = pd.read_csv(INPUT_CSV, nrows=5)
    if HADM_COL in sample.columns:
        usecols.append(HADM_COL)
except Exception:
    pass

reader = pd.read_csv(INPUT_CSV, usecols=usecols, chunksize=CHUNKSIZE, iterator=True, low_memory=False)

rows_processed = 0
for i, chunk in enumerate(reader):
    print(f"Processing chunk {i+1} ...")
    # ensure types
    chunk[TEXT_COL] = chunk[TEXT_COL].astype(str).fillna("")
    # parse times (best-effort)
    if CHARTTIME_COL in chunk.columns:
        chunk[CHARTTIME_COL] = pd.to_datetime(chunk[CHARTTIME_COL], errors='coerce')
    if STORETIME_COL in chunk.columns:
        chunk[STORETIME_COL] = pd.to_datetime(chunk[STORETIME_COL], errors='coerce')

    # quick vectorized filter: keep rows that contain any of the phenotype keywords (big OR)
    combined_pattern = "|".join("|".join(pats) for pats in PHENOTYPE_PATTERNS.values())
    mask = chunk[TEXT_COL].str.contains(combined_pattern, case=False, na=False, regex=True)
    filtered = chunk[mask].copy()
    print(f"  rows in chunk: {len(chunk)}, rows with any keyword: {len(filtered)}")

    # iterate filtered rows (now smaller)
    for idx, row in filtered.iterrows():
        rows_processed += 1
        subj = int(row[SUBJ_COL])
        text = row[TEXT_COL]
        mention_time = row[CHARTTIME_COL] if CHARTTIME_COL in filtered.columns and not pd.isna(row.get(CHARTTIME_COL)) else (row.get(STORETIME_COL) if STORETIME_COL in filtered.columns else pd.NaT)

        # check each phenotype
        for ph, regex_list in phen_regexes.items():
            found = False
            for reg in regex_list:
                for m in reg.finditer(text):
                    start = m.start()
                    if is_negated_around(text, start):
                        continue
                    # valid mention
                    update_pheno_stats(subj, ph, mention_time)
                    found = True
                    # break to count each match separately or break to count one per row?
                    # We'll count all non-negated matches; if you prefer one per row, uncomment break.
                    # break
                # if found: break
    print(f"  processed filtered rows so far: {rows_processed}")

# convert stats to dataframe rows
rows_out = []
for subj, ph_dict in stats.items():
    row = {"subject_id": subj}
    for ph, entry in ph_dict.items():
        row[f"{ph}_any"] = int(entry["any_mention"])
        row[f"{ph}_count"] = int(entry["mention_count"])
        # times as ISO strings or NaT
        row[f"{ph}_first_time"] = entry["first_mention_time"]
        row[f"{ph}_last_time"]  = entry["last_mention_time"]
    rows_out.append(row)

pheno_df = pd.DataFrame(rows_out)

# Save results
os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)
pheno_df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved radiology phenotype features to: {OUTPUT_CSV}")
print("Subjects covered:", len(pheno_df), "Unique phenotypes cols:", [c for c in pheno_df.columns if c!='subject_id'])


Processing chunk 1 ...
  rows in chunk: 50000, rows with any keyword: 5343
  processed filtered rows so far: 5343
Processing chunk 2 ...
  rows in chunk: 50000, rows with any keyword: 5906
  processed filtered rows so far: 11249
Processing chunk 3 ...
  rows in chunk: 50000, rows with any keyword: 5535
  processed filtered rows so far: 16784
Processing chunk 4 ...
  rows in chunk: 50000, rows with any keyword: 5801
  processed filtered rows so far: 22585
Processing chunk 5 ...
  rows in chunk: 50000, rows with any keyword: 5653
  processed filtered rows so far: 28238
Processing chunk 6 ...
  rows in chunk: 50000, rows with any keyword: 6089
  processed filtered rows so far: 34327
Processing chunk 7 ...
  rows in chunk: 50000, rows with any keyword: 5682
  processed filtered rows so far: 40009
Processing chunk 8 ...
  rows in chunk: 50000, rows with any keyword: 5842
  processed filtered rows so far: 45851
Processing chunk 9 ...
  rows in chunk: 50000, rows with any keyword: 5695
  proc

In [15]:
# produce_compact_radiology_summary.py

# paths - adjust if needed
RADIO_FULL = "../data/processed/radiology_phenotypes_by_subject.csv"   # produced by your extractor
COHORT = "../data/processed/final_hosp_dataset.csv"                   # to ensure same subjects as cohort
RADIO_COMPACT_OUT = "../data/processed/radiology_compact_summary.csv"

# 1. load files
rad = pd.read_csv(RADIO_FULL, low_memory=False)
cohort = pd.read_csv(COHORT, usecols=['subject_id'])

# ensure subject_id int
rad['subject_id'] = rad['subject_id'].astype(int)
cohort['subject_id'] = cohort['subject_id'].astype(int)

# 2. identify phenotype groups in rad (cols like <ph>_any, <ph>_count, <ph>_first_time, <ph>_last_time)
cols = rad.columns.tolist()
any_cols = [c for c in cols if c.endswith('_any')]
count_cols = [c for c in cols if c.endswith('_count')]
first_time_cols = [c for c in cols if c.endswith('_first_time')]
last_time_cols = [c for c in cols if c.endswith('_last_time')]

# If nothing found, raise informative error
if len(any_cols) == 0 and len(count_cols) == 0:
    raise ValueError(f"No phenotype columns found in {RADIO_FULL}. Expected columns like '<phen>_any' or '<phen>_count'")

# 3. Normalize types: ensure times parsed
for tcol in first_time_cols + last_time_cols:
    if tcol in rad.columns:
        rad[tcol] = pd.to_datetime(rad[tcol], errors='coerce')

# 4. Aggregate per subject into a single HTN-related radiology summary
# rad_any_mention: True if ANY phenotype_any == 1 (or True)
# rad_mention_count: sum of all phenotype counts (treat NaN as 0)
# rad_first_mention_time: earliest non-null first_time across phenotypes
# rad_last_mention_time: latest non-null last_time across phenotypes

def compute_compact(df):
    out = {}
    # any: if any of the *_any columns is 1/True
    if any_cols:
        # treat 'any' columns as truthy if 1/True
        any_bool = df[any_cols].fillna(0).astype(int).sum(axis=1) > 0
        # Since each row here is per subject, we reduce by logical OR:
        out_any = any_bool.any()
    else:
        out_any = False

    # mention_count: sum of counts
    if count_cols:
        # sum across all count cols treating NaN as 0
        out_count = int(df[count_cols].fillna(0).sum(axis=1).sum())
    else:
        out_count = 0

    # first_time: min among available first_times
    if first_time_cols:
        first_times = pd.to_datetime(df[first_time_cols].stack().droplevel(1)).sort_values()
        out_first = first_times.min() if not first_times.empty else pd.NaT
    else:
        out_first = pd.NaT

    # last_time: max among available last_times
    if last_time_cols:
        last_times = pd.to_datetime(df[last_time_cols].stack().droplevel(1)).sort_values()
        out_last = last_times.max() if not last_times.empty else pd.NaT
    else:
        out_last = pd.NaT

    return pd.Series({
        'rad_any_mention': out_any,
        'rad_mention_count': out_count if out_count > 0 else np.nan,
        'rad_first_mention_time': out_first,
        'rad_last_mention_time': out_last
    })

# group & aggregate (in case rad has only one row per subject this still works)
compact_rows = []
for subj, g in rad.groupby('subject_id'):
    s = compute_compact(g)
    s['subject_id'] = subj
    compact_rows.append(s)

rad_compact = pd.DataFrame(compact_rows)

# 5. Ensure every cohort subject is present: left-join cohort (keeps all subjects)
rad_compact = cohort.merge(rad_compact, on='subject_id', how='left')

# 6. Convert rad_any_mention to True/NaN semantics (like your discharge file)
# If rad_any_mention is True -> keep True; if False or NaN -> set NaN (so NA means no note info)
rad_compact['rad_any_mention'] = rad_compact['rad_any_mention'].map({True: True, False: np.nan})

# Keep rad_mention_count as numeric (NaN if missing)
rad_compact['rad_mention_count'] = pd.to_numeric(rad_compact['rad_mention_count'], errors='coerce')

# 7. Save in same style as discharge
os.makedirs(os.path.dirname(RADIO_COMPACT_OUT), exist_ok=True)
rad_compact.to_csv(RADIO_COMPACT_OUT, index=False)
print("Saved compact radiology summary to:", RADIO_COMPACT_OUT)
print("Example rows:")
print(rad_compact.head())
print("\nCoverage:")
print("Subjects with any radiology mention (non-null):", rad_compact['rad_any_mention'].notna().mean()*100, "%")


Saved compact radiology summary to: ../data/processed/radiology_compact_summary.csv
Example rows:
   subject_id rad_any_mention  rad_mention_count rad_first_mention_time  \
0    10000032             NaN                NaN                    NaT   
1    10000068             NaN                NaN                    NaT   
2    10000084             NaN                NaN                    NaT   
3    10000108             NaN                NaN                    NaT   
4    10000117             NaN                NaN                    NaT   

  rad_last_mention_time  
0                   NaT  
1                   NaT  
2                   NaT  
3                   NaT  
4                   NaT  

Coverage:
Subjects with any radiology mention (non-null): 20.10623594135342 %


In [16]:
print(rad_compact.columns)
rad_compact.head()

Index(['subject_id', 'rad_any_mention', 'rad_mention_count',
       'rad_first_mention_time', 'rad_last_mention_time'],
      dtype='object')


,subject_id,rad_any_mention,rad_mention_count,rad_first_mention_time,rad_last_mention_time
0,10000032,NaN,NaN,NaT,NaT
1,10000068,NaN,NaN,NaT,NaT
2,10000084,NaN,NaN,NaT,NaT
3,10000108,NaN,NaN,NaT,NaT
4,10000117,NaN,NaN,NaT,NaT


In [5]:
rad_compact = pd.read_csv("../data/processed/radiology_compact_summary.csv")

missing_values_prop = rad_compact.isnull().mean()
a = missing_values_prop*100.0
print("missings proportion:\n", a)

print(len(np.unique(rad_compact['subject_id'])))
print(rad_compact['rad_any_mention'].value_counts())

missings proportion:
 subject_id                 0.000000
rad_any_mention           79.893764
rad_mention_count         79.893764
rad_first_mention_time    79.893764
rad_last_mention_time     79.893764
dtype: float64
36899
rad_any_mention
True    7419
Name: count, dtype: int64


## Optional: inspect nrows of huge csv file

In [6]:
radiology_sample = pd.read_csv("E:/Chrome Dls/MIMIC_IV_Note/note/radiology.csv", nrows=100)
radiology_sample.to_csv(
    "../data/processed/radiology_sample.csv",
    index=False,
    quoting=1,        # csv.QUOTE_ALL
    escapechar='\\'   # escape problematic characters
)

radiology_sample.head()

,note_id,subject_id,hadm_id,note_type,note_seq,charttime,storetime,text
0,10000032-RR-14,10000032,22595853.0,RR,14,2180-05-06 21:19:00,2180-05-06 23:32:00,EXAMINATION: CHEST (PA AND LAT)\n\nINDICATION...
1,10000032-RR-15,10000032,22595853.0,RR,15,2180-05-06 23:00:00,2180-05-06 23:26:00,EXAMINATION: LIVER OR GALLBLADDER US (SINGLE ...
2,10000032-RR-16,10000032,22595853.0,RR,16,2180-05-07 09:55:00,2180-05-07 11:15:00,"INDICATION: ___ HCV cirrhosis c/b ascites, hi..."
3,10000032-RR-18,10000032,NaN,RR,18,2180-06-03 12:46:00,2180-06-03 14:01:00,EXAMINATION: Ultrasound-guided paracentesis.\...
4,10000032-RR-20,10000032,NaN,RR,20,2180-07-08 13:18:00,2180-07-08 14:15:00,EXAMINATION: Paracentesis\n\nINDICATION: ___...


In [9]:
print(radiology_sample.columns)
radiology_sample.head()

Index(['note_id', 'subject_id', 'hadm_id', 'note_type', 'note_seq',
       'charttime', 'storetime', 'text'],
      dtype='object')


,note_id,subject_id,hadm_id,note_type,note_seq,charttime,storetime,text
0,10000032-RR-14,10000032,22595853.0,RR,14,2180-05-06 21:19:00,2180-05-06 23:32:00,EXAMINATION: CHEST (PA AND LAT)\n\nINDICATION...
1,10000032-RR-15,10000032,22595853.0,RR,15,2180-05-06 23:00:00,2180-05-06 23:26:00,EXAMINATION: LIVER OR GALLBLADDER US (SINGLE ...
2,10000032-RR-16,10000032,22595853.0,RR,16,2180-05-07 09:55:00,2180-05-07 11:15:00,"INDICATION: ___ HCV cirrhosis c/b ascites, hi..."
3,10000032-RR-18,10000032,NaN,RR,18,2180-06-03 12:46:00,2180-06-03 14:01:00,EXAMINATION: Ultrasound-guided paracentesis.\...
4,10000032-RR-20,10000032,NaN,RR,20,2180-07-08 13:18:00,2180-07-08 14:15:00,EXAMINATION: Paracentesis\n\nINDICATION: ___...


## Optional: compare radiology features against Labels

In [8]:
radio_df = pd.read_csv("../data/processed/radiology_phenotypes_by_subject.csv")
patients_df = pd.read_csv("../data/processed/patients.csv")

In [ ]:
print(len(np.unique(radio_df['subject_id']))) # 46433
print(len(np.unique(patients_df['subject_id']))) #36899

46433
36899


In [10]:

# paths (adjust if needed)
RADIO_COMPACT = "../data/processed/radiology_compact_summary.csv"
RADIO_FULL = "../data/processed/radiology_phenotypes_by_subject.csv"
PATIENTS = "../data/processed/patients.csv"

# load patients & labels
patients_df = pd.read_csv(PATIENTS, usecols=['subject_id','label'])
patients_df['subject_id'] = patients_df['subject_id'].astype(int)

# attempt to load compact radiology file first
if os.path.exists(RADIO_COMPACT):
    rad_df = pd.read_csv(RADIO_COMPACT)
    # normalize column name to rad_any_mention if different
    possible_any_cols = [c for c in rad_df.columns if c.lower().endswith('any_mention') or c.lower().endswith('_any')]
    if 'rad_any_mention' in rad_df.columns:
        rad_any_col = 'rad_any_mention'
    elif len(possible_any_cols) > 0:
        rad_any_col = possible_any_cols[0]
        # rename for consistency
        rad_df = rad_df.rename(columns={rad_any_col: 'rad_any_mention'})
        rad_any_col = 'rad_any_mention'
    else:
        # fallback: if compact exists but no any-col, create one if count col exists
        count_cols = [c for c in rad_df.columns if c.endswith('_count')]
        if len(count_cols) > 0:
            rad_df['rad_any_mention'] = rad_df[count_cols].fillna(0).sum(axis=1) > 0
        else:
            rad_df['rad_any_mention'] = np.nan
    # ensure subject_id int
    rad_df['subject_id'] = rad_df['subject_id'].astype(int)
    print(f"Using compact radiology summary: {RADIO_COMPACT}, rows: {len(rad_df)}")
else:
    # fallback: load full phenotype-by-subject and compute aggregated any-flag
    if not os.path.exists(RADIO_FULL):
        raise FileNotFoundError(f"Neither {RADIO_COMPACT} nor {RADIO_FULL} found. Place one of them in the path.")
    rad_full = pd.read_csv(RADIO_FULL, low_memory=False)
    rad_full['subject_id'] = rad_full['subject_id'].astype(int)
    # find any-columns (e.g., pulmonary_edema_any, cardiomegaly_any, etc.)
    any_cols = [c for c in rad_full.columns if c.endswith('_any')]
    if len(any_cols) == 0:
        raise ValueError(f"No '*_any' columns found in {RADIO_FULL}; can't build aggregated rad_any_mention.")
    # build aggregated rad_any_mention: True if any of the *_any columns == 1/True
    # some *_any may be 0/1 or True/False; normalize by treating truthy values as True
    def col_truthy(series):
        return series.fillna(0).astype(int).clip(0,1)
    # compute per-subject OR (note: rad_full may already be one-row-per-subject)
    agg = rad_full.copy()
    for c in any_cols:
        # coerce to 0/1 numeric where possible
        try:
            agg[c] = col_truthy(agg[c])
        except Exception:
            agg[c] = agg[c].astype(str).str.lower().isin(['1','true','t','yes','y']).astype(int)
    agg['rad_any_mention'] = (agg[any_cols].sum(axis=1) > 0)
    rad_df = agg[['subject_id','rad_any_mention']].drop_duplicates().reset_index(drop=True)
    print(f"Built aggregated rad_any_mention from full radiology phenotypes: {RADIO_FULL}, rows: {len(rad_df)}")

# merge label + rad feature (left join on patients to keep same cohort)
merged = pd.merge(patients_df, rad_df[['subject_id','rad_any_mention']], on='subject_id', how='left')

# normalize rad_any_mention to boolean True for explicit mentions only
# treat values 1/True/'True' as True; leave NaN as NaN (missing)
def is_explicit_true(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, (bool, np.bool_)):
        return bool(x)
    try:
        if int(x) == 1:
            return True
    except Exception:
        pass
    if str(x).lower() in ('true','t','yes','y','1'):
        return True
    return False

merged['rad_any_mention_explicit'] = merged['rad_any_mention'].apply(is_explicit_true)

# Now compute counts
total_htn = merged[merged['label'] == 1].shape[0]
htn_with_mention = merged[(merged['label'] == 1) & (merged['rad_any_mention_explicit'] == True)].shape[0]

total_nonhtn = merged[merged['label'] == 0].shape[0]
nonhtn_with_mention = merged[(merged['label'] == 0) & (merged['rad_any_mention_explicit'] == True)].shape[0]

# percentages
overlap_ratio = (htn_with_mention / total_htn * 100) if total_htn > 0 else np.nan
precision_like = (htn_with_mention / (htn_with_mention + nonhtn_with_mention) * 100) if (htn_with_mention + nonhtn_with_mention) > 0 else np.nan
coverage_notes = merged['rad_any_mention'].notna().mean() * 100  # percent subjects with any radiology info (not NaN)

print("\n=== Radiology vs Label overlap ===")
print(f"Total labeled positive (label=1): {total_htn}")
print(f"Labeled positives with explicit radiology mention: {htn_with_mention}")
print(f"Overlap ratio (percent of positives with rad mention): {overlap_ratio:.2f}%")
print()
print(f"Total labeled negative (label=0): {total_nonhtn}")
print(f"Labeled negatives with explicit rad mention: {nonhtn_with_mention}")
print()
print(f"Among all subjects, percent having any radiology feature present (not NaN): {coverage_notes:.2f}%")
print(f"Precision-like (fraction of rad mentions that are label=1): {precision_like:.2f}%")

# quick samples of mismatches:
sample_pos_no_rad = merged[(merged['label']==1) & (merged['rad_any_mention_explicit'] != True)].sample(min(10, merged[(merged['label']==1) & (merged['rad_any_mention_explicit'] != True)].shape[0]), random_state=42)['subject_id'].tolist()
sample_neg_with_rad = merged[(merged['label']==0) & (merged['rad_any_mention_explicit'] == True)].sample(min(10, merged[(merged['label']==0) & (merged['rad_any_mention_explicit'] == True)].shape[0]), random_state=42)['subject_id'].tolist()

print("\nExample subject_ids (label=1 but NO explicit rad mention) sample:", sample_pos_no_rad)
print("Example subject_ids (label=0 but HAVE explicit rad mention) sample:", sample_neg_with_rad)

# Save merged summary for inspection if desired
OUT_SUM = "../reports/radiology_label_overlap_summary.csv"
merged.to_csv(OUT_SUM, index=False)
print(f"\nSaved merged summary to: {OUT_SUM}")


Using compact radiology summary: ../data/processed/radiology_compact_summary.csv, rows: 36899

=== Radiology vs Label overlap ===
Total labeled positive (label=1): 18485
Labeled positives with explicit radiology mention: 5773
Overlap ratio (percent of positives with rad mention): 31.23%

Total labeled negative (label=0): 18414
Labeled negatives with explicit rad mention: 1646

Among all subjects, percent having any radiology feature present (not NaN): 20.11%
Precision-like (fraction of rad mentions that are label=1): 77.81%

Example subject_ids (label=1 but NO explicit rad mention) sample: [10121750, 11119218, 10529456, 10903221, 11554081, 10566969, 10472051, 10218242, 10834232, 10119732]
Example subject_ids (label=0 but HAVE explicit rad mention) sample: [10724933, 11567778, 10783654, 11168367, 11287673, 10229974, 11544133, 10853018, 11250079, 10945254]

Saved merged summary to: ../reports/radiology_label_overlap_summary.csv


If the overlap ratio is 100%, it means every hypertensive patient has a mention in the notes → ⚠️ potential leakage (your NLP feature may be duplicating your label).

If it’s much less (e.g., 30–60%), it’s fine — this means the note feature adds real-world complementary info.

## Optional: 
- loads discharge_htn_features.csv, admissions.csv, and your final_hosp_dataset.csv (to get labels),

- computes, per subject_id, the earliest admission time and the latest discharge time,

- compares htn_first_mention_time / htn_last_mention_time to these admission times,

- produces summary statistics showing how many note mentions occur after the associated hospitalization (possible leakage),

- saves a CSV with per-subject flags you can inspect.

In [10]:
# leakage_timing_check.py
import pandas as pd
import numpy as np
import os

# ---------------- CONFIG ----------------
DISCH_FEAT = "../data/processed/discharge_htn_features.csv"   # has subject_id, htn_first_mention_time, htn_last_mention_time
ADMISSIONS_CSV = "E:/Chrome Dls/MIMIC_IV_Core/hosp/admissions.csv"               # standard MIMIC admissions table (hadm-level)
FINAL_HOSP = "../data/processed/final_hosp_dataset.csv"      # your structured dataset with labels (subject_id, label)
OUT_SUMMARY = "../data/processed/notes_timing_leakage_summary.csv"
# ----------------------------------------

# 1) Load files
print("Loading files...")
disch = pd.read_csv(DISCH_FEAT, parse_dates=['htn_first_mention_time', 'htn_last_mention_time'], low_memory=False)
admissions = pd.read_csv(ADMISSIONS_CSV, usecols=['subject_id', 'hadm_id', 'admittime', 'dischtime'], parse_dates=['admittime','dischtime'], low_memory=False)
final_hosp = pd.read_csv(FINAL_HOSP, usecols=['subject_id','label'], low_memory=False)  # label column name assumed 'label'

print(f"disch rows: {len(disch)}, admissions rows: {len(admissions)}, final_hosp rows: {len(final_hosp)}")

# 2) Compute per-subject admission windows: earliest admit, latest discharge
adm_agg = admissions.groupby('subject_id').agg(
    earliest_admit = ('admittime', 'min'),
    latest_discharge = ('dischtime', 'max'),
    n_admissions = ('hadm_id', 'nunique')
).reset_index()

print("Aggregated admissions (per subject) sample:")
print(adm_agg.head())

# 3) Merge discharge features with admissions aggregation and labels
merged = pd.merge(final_hosp, disch, on='subject_id', how='left')   # keep only your cohort subjects
merged = pd.merge(merged, adm_agg, on='subject_id', how='left')      # may be NaT for patients without admission rows

# 4) Ensure datetime dtypes
for col in ['htn_first_mention_time','htn_last_mention_time','earliest_admit','latest_discharge']:
    if col in merged.columns:
        merged[col] = pd.to_datetime(merged[col], errors='coerce')

# 5) Create temporal flags (per subject)
# - mention_exists: whether any mention was detected (non-null first or last)
merged['mention_exists'] = (~merged['htn_first_mention_time'].isna()) | (~merged['htn_last_mention_time'].isna())

# Use last_mention if present, else first_mention for timing checks
merged['mention_time_used'] = merged['htn_last_mention_time'].fillna(merged['htn_first_mention_time'])

# Flags:
# - mention_before_any_admit: mention_time_used <= earliest_admit  (meaning mention is before first admission in admissions table)
# - mention_during_admission_window: earliest_admit <= mention_time_used <= latest_discharge
# - mention_after_latest_discharge: mention_time_used > latest_discharge
# - no_admission_info: earliest_admit is NaT (no matching admissions in admissions.csv)
merged['no_admission_info'] = merged['earliest_admit'].isna()
merged['mention_before_any_admit'] = np.where(merged['mention_time_used'].notna() & merged['earliest_admit'].notna(),
                                              merged['mention_time_used'] <= merged['earliest_admit'],
                                              False)
merged['mention_during_admission_window'] = np.where(merged['mention_time_used'].notna() & merged['earliest_admit'].notna() & merged['latest_discharge'].notna(),
                                                    (merged['mention_time_used'] >= merged['earliest_admit']) & (merged['mention_time_used'] <= merged['latest_discharge']),
                                                    False)
merged['mention_after_latest_discharge'] = np.where(merged['mention_time_used'].notna() & merged['latest_discharge'].notna(),
                                                   merged['mention_time_used'] > merged['latest_discharge'],
                                                   False)

# 6) Summaries to quantify potential leakage
total_subjects = len(merged)
mention_exists_count = merged['mention_exists'].sum()
no_adm_info_count = merged['no_admission_info'].sum()

# Among subjects with mentions:
m = merged[merged['mention_exists']]
m_count = len(m)
m_before_admit = m['mention_before_any_admit'].sum()
m_during = m['mention_during_admission_window'].sum()
m_after = m['mention_after_latest_discharge'].sum()
m_no_adm = m['no_admission_info'].sum()

print("\n--- Overall counts ---")
print(f"Total subjects in final_hosp: {total_subjects}")
print(f"Subjects with any mention (note-derived): {mention_exists_count} ({mention_exists_count/total_subjects*100:.2f}%)")
print(f"Subjects without admission records (no_admission_info): {no_adm_info_count} ({no_adm_info_count/total_subjects*100:.2f}%)")

print("\n--- Among subjects with mentions ---")
print(f"Total with mention: {m_count}")
print(f"Mentions before first admission: {m_before_admit} ({m_before_admit/m_count*100:.2f}%)")
print(f"Mentions during admission windows: {m_during} ({m_during/m_count*100:.2f}%)")
print(f"Mentions after latest discharge: {m_after} ({m_after/m_count*100:.2f}%)")
print(f"Mentions but no admission info: {m_no_adm} ({m_no_adm/m_count*100:.2f}%)")

# 7) Now, check leakage specifically w.r.t. your labels
# How many label=1 patients have mention; of those, how many mentions happen after discharge?
lbl = merged[merged['label'] == 1]
lbl_total = len(lbl)
lbl_with_mention = lbl['mention_exists'].sum()
lbl_with_after = lbl['mention_after_latest_discharge'].sum()

print("\n--- Among labeled positives (label==1) ---")
print(f"Total labeled positive: {lbl_total}")
print(f"Labeled positives with any mention in notes: {lbl_with_mention} ({lbl_with_mention / lbl_total * 100:.2f}%)")
print(f"Labeled positives whose mention occurs AFTER latest discharge: {lbl_with_after} ({lbl_with_after / max(1,lbl_with_mention) * 100:.2f}% of those with mention)")

# 8) Save per-subject CSV for inspection
os.makedirs(os.path.dirname(OUT_SUMMARY), exist_ok=True)
cols_to_save = [
    'subject_id','label','mention_exists','htn_first_mention_time','htn_last_mention_time','mention_time_used',
    'earliest_admit','latest_discharge','n_admissions',
    'no_admission_info','mention_before_any_admit','mention_during_admission_window','mention_after_latest_discharge'
]
merged[cols_to_save].to_csv(OUT_SUMMARY, index=False)
print(f"\nSaved per-subject timing summary to: {OUT_SUMMARY}")

# 9) Short guidance on interpretation
print("\nINTERPRETATION GUIDANCE:")
print("- If a large fraction of mentions are AFTER latest_discharge, that suggests note-derived features may be written after hospitalization and could leak outcome information.")
print("- If most mentions are DURING admission window or BEFORE earliest_admit, they are more likely predictive or represent prior history (safer).")
print("- Use the saved CSV to inspect individual examples where mention occurs after discharge (look at mention_time_used, latest_discharge, and raw notes if needed).")


Loading files...
disch rows: 86814, admissions rows: 546028, final_hosp rows: 36899
Aggregated admissions (per subject) sample:
   subject_id      earliest_admit    latest_discharge  n_admissions
0    10000032 2180-05-06 22:23:00 2180-08-07 17:50:00             4
1    10000068 2160-03-03 23:16:00 2160-03-04 06:26:00             1
2    10000084 2160-11-21 01:56:00 2160-12-28 16:07:00             2
3    10000108 2163-09-27 23:17:00 2163-09-28 09:04:00             1
4    10000117 2181-11-15 02:05:00 2183-09-21 16:30:00             2

--- Overall counts ---
Total subjects in final_hosp: 36899
Subjects with any mention (note-derived): 14392 (39.00%)
Subjects without admission records (no_admission_info): 0 (0.00%)

--- Among subjects with mentions ---
Total with mention: 14392
Mentions before first admission: 186 (1.29%)
Mentions during admission windows: 14205 (98.70%)
Mentions after latest discharge: 2 (0.01%)
Mentions but no admission info: 0 (0.00%)

--- Among labeled positives (label==